In [ ]:
import requests
import json
import base64
import time
import notebookutils
import sempy.fabric as fabric

In [ ]:
# ── Cell 1: Config  (USER EDITS THIS) ───────────────────────
LAKEHOUSE_NAME  = "MyLakehouse"
PIPELINE_NAME   = "MyPipeline"
WORKSPACE_ID    = fabric.get_workspace_id()
TOKEN           = notebookutils.credentials.getToken("pbi")

# API keys — user fills these in
API_KEY         = "your-api-key-here"
API_SECRET      = "your-api-secret-here"

GITHUB_RAW_BASE  = "https://raw.githubusercontent.com/ASM205/Stocks/main"
GITHUB_API_BASE  = "https://api.github.com/repos/ASM205/Stocks/contents"

In [ ]:
HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type" : "application/json"
}
print("✅ Config loaded")

In [ ]:
import requests, json, base64, time
import notebookutils, sempy.fabric as fabric

print("\n── Discovering notebooks from GitHub ───────────────────")

gh_resp = requests.get(
    f"{GITHUB_API_BASE}/notebooks",
    headers=GITHUB_HEADERS
)

if gh_resp.status_code != 200:
    raise RuntimeError(f"Could not read GitHub notebooks folder: {gh_resp.status_code} {gh_resp.text}")

all_files = gh_resp.json()

# Get all .ipynb files sorted by name, excluding Setup.ipynb
NOTEBOOK_FILES = sorted([
    f for f in all_files
    if f["name"].endswith(".ipynb") and f["name"].lower() != "setup.ipynb"
], key=lambda x: x["name"])

print(f"  Found {len(NOTEBOOK_FILES)} notebooks:")
for f in NOTEBOOK_FILES:
    print(f"    - {f['name']}")


# ── Helper: convert .ipynb JSON → Fabric .py source ─────────
def ipynb_to_fabric_py(ipynb_json: dict) -> str:
    """Extract code cells from a .ipynb and format as Fabric .py source."""
    lines = [
        "# Fabric notebook source\n",
        "\n",
        "# METADATA ********************\n",
        "\n",
        "# META {\n",
        '# META   "kernel_info": {"name": "synapse_pyspark"},\n',
        '# META   "language_info": {"name": "python"}\n',
        "# META }\n",
        "\n",
    ]
    for cell in ipynb_json.get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        lines.append("# CELL ********************\n\n")
        source = cell.get("source", [])
        # source can be a list of strings or a single string
        if isinstance(source, list):
            lines.extend(source)
        else:
            lines.append(source)
        lines.append("\n\n")
    return "".join(lines)


# ── Helper: wait for item to appear in workspace ─────────────
def wait_for_item(display_name: str, item_type: str, retries=15, delay=5) -> str:
    for i in range(retries):
        items = requests.get(
            f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}/items",
            headers=HEADERS
        ).json().get("value", [])
        match = next(
            (x for x in items if x["displayName"] == display_name and x["type"] == item_type),
            None
        )
        if match:
            return match["id"]
        print(f"  Waiting for {item_type} '{display_name}'... ({i+1}/{retries})")
        time.sleep(delay)
    raise RuntimeError(f"Timed out waiting for {item_type} '{display_name}'")


def poll_operation(operation_url: str, label: str, retries=20, delay=3):
    for _ in range(retries):
        time.sleep(delay)
        resp  = requests.get(operation_url, headers=HEADERS)
        state = resp.json().get("status", "")
        print(f"  [{label}] → {state}")
        if state == "Succeeded":
            return
        if state == "Failed":
            raise RuntimeError(f"{label} failed: {resp.json()}")
    raise RuntimeError(f"{label} polling timed out")


def delete_if_exists(display_name: str, item_type: str):
    items = requests.get(
        f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}/items",
        headers=HEADERS
    ).json().get("value", [])
    for item in items:
        if item["displayName"] == display_name and item["type"] == item_type:
            requests.delete(
                f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}/items/{item['id']}",
                headers=HEADERS
            )
            print(f"  Deleted existing {item_type}: {display_name}")
            time.sleep(3)

In [ ]:
print("\n── Creating Lakehouse ──────────────────────────────────")
delete_if_exists(LAKEHOUSE_NAME, "Lakehouse")

lh_resp = requests.post(
    f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}/items",
    headers=HEADERS,
    json={"displayName": LAKEHOUSE_NAME, "type": "Lakehouse"}
)
if lh_resp.status_code == 202:
    op_id = lh_resp.headers.get("x-ms-operation-id", "")
    poll_operation(f"https://api.fabric.microsoft.com/v1/operations/{op_id}", "Lakehouse")

LAKEHOUSE_ID = wait_for_item(LAKEHOUSE_NAME, "Lakehouse")
print(f"✅ Lakehouse ID: {LAKEHOUSE_ID}")

In [ ]:
print("\n── Creating Notebooks ──────────────────────────────────")

notebook_ids = {}  # { display_name: id }
first_notebook = True

for nb_file in NOTEBOOK_FILES:
    nb_name = nb_file["name"].replace(".ipynb", "")
    print(f"\n  Creating: {nb_name}")
    delete_if_exists(nb_name, "Notebook")

    # Fetch raw .ipynb from GitHub
    raw_url  = f"{GITHUB_RAW_BASE}/notebooks/{nb_file['name']}"
    raw_resp = requests.get(raw_url, headers=GITHUB_HEADERS)
    if raw_resp.status_code != 200:
        raise RuntimeError(f"Could not fetch {nb_file['name']}: {raw_resp.status_code}")

    ipynb_json = raw_resp.json()

    # Convert to Fabric .py format
    py_source = ipynb_to_fabric_py(ipynb_json)

    # Inject API keys into first notebook only
    if first_notebook:
        py_source = py_source.replace('API_KEY      = ""', f'API_KEY      = "{API_KEY}"') \
                             .replace('API_SECRET   = ""', f'API_SECRET   = "{API_SECRET}"') \
                             .replace('API_BASE_URL = ""', f'API_BASE_URL = "{API_BASE_URL}"')
        print(f"  ✅ API keys injected")
        first_notebook = False

    payload = base64.b64encode(py_source.encode("utf-8")).decode("utf-8")

    resp = requests.post(
        f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}/items",
        headers=HEADERS,
        json={
            "displayName": nb_name,
            "type"        : "Notebook",
            "definition"  : {
                "parts": [{
                    "path"       : "notebook-content.py",
                    "mimeType"   : "text/x-python",
                    "payloadType": "InlineBase64",
                    "payload"    : payload
                }]
            }
        }
    )
    print(f"  Status: {resp.status_code}")

    if resp.status_code == 202:
        op_id = resp.headers.get("x-ms-operation-id", "")
        poll_operation(f"https://api.fabric.microsoft.com/v1/operations/{op_id}", nb_name)

    time.sleep(5)
    nb_id = wait_for_item(nb_name, "Notebook")
    notebook_ids[nb_name] = nb_id
    print(f"  ✅ {nb_name}: {nb_id}")



In [ ]:
print("\n── Creating Pipeline ────────────────────────────────────")
delete_if_exists(PIPELINE_NAME, "DataPipeline")

nb_list     = list(notebook_ids.items())
activities  = []
for i, (nb_name, nb_id) in enumerate(nb_list):
    activities.append({
        "name"          : f"Run_{nb_name}",
        "type"          : "TridentNotebook",
        "dependsOn"     : [] if i == 0 else [{
            "activity"            : f"Run_{nb_list[i-1][0]}",
            "dependencyConditions": ["Succeeded"]
        }],
        "typeProperties": {
            "notebookId" : nb_id,
            "workspaceId": WORKSPACE_ID
        }
    })

pipeline_payload = base64.b64encode(json.dumps({
    "name"      : PIPELINE_NAME,
    "properties": {"activities": activities}
}).encode()).decode()

pl_resp = requests.post(
    f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}/items",
    headers=HEADERS,
    json={
        "displayName": PIPELINE_NAME,
        "type"        : "DataPipeline",
        "definition"  : {
            "parts": [{
                "path"       : "pipeline-content.json",
                "mimeType"   : "application/json",
                "payloadType": "InlineBase64",
                "payload"    : pipeline_payload
            }]
        }
    }
)
print(f"Pipeline status: {pl_resp.status_code}")

if pl_resp.status_code == 202:
    op_id = pl_resp.headers.get("x-ms-operation-id", "")
    poll_operation(f"https://api.fabric.microsoft.com/v1/operations/{op_id}", "Pipeline")

time.sleep(10)
PIPELINE_ID = wait_for_item(PIPELINE_NAME, "DataPipeline")
print(f"✅ Pipeline ID: {PIPELINE_ID}")
